# 02 · Trade flows — pathway-based trade tensor

**Data stream:** trade flows between SInAS locations.
**Output feeds:** the trade stream (Stream D) of the model:
`trade_channels.pt`.

---

## What this notebook does

Bilateral trade from **CEPII BACI** (HS 1992 nomenclature, every year
1995–2024) is grouped into **channels that match invasion pathways**,
averaged over the years, and re-mapped from BACI countries to the 289
SInAS locations. The result is one exporter × importer matrix per
channel.

### Channels

Channels follow the CBD pathway categories (Hulme et al. 2008; CBD 2014):
*escape / release* (live organisms traded on purpose), *contaminant* (on
or in a commodity) and *stowaway* (on the transport vector or packaging).
Each 6-digit HS product belongs to at most one channel.

| Channel | HS 1992 codes | Pathway | Why |
|---|---|---|---|
| `plants_for_planting` | 0601, 0602, 1209 | escape / contaminant | live plants, bulbs, seeds for sowing: the main pathway for plant pests and invasive plants |
| `cut_flowers_foliage` | 0603, 0604 | contaminant | insects, mites, pathogens on fresh flowers |
| `live_terrestrial_animals` | 0101–0106 | escape / release | pets, livestock, other live animals |
| `live_aquatic_animals` | 0301, 03062, 0307 | escape / contaminant | aquarium fish, live crustaceans, molluscs (oysters carry hitchhikers) |
| `fresh_fruit_vegetables` | 0701–0709, 0714, 0801–0810 | contaminant | fruit flies, other insects and pathogens on fresh produce |
| `grain_seeds_legumes` | 10, 12 (excl. 1209), 0713 | contaminant | weed seeds, stored-product pests |
| `roundwood_sawnwood` | 4401, 4403, 4404, 4406, 4407 | contaminant | bark and wood-boring insects, fungi |
| `wood_packaging` | 4415, 4416 | stowaway | pallets, crates, casks: wood-packaging pests |
| `containerised_goods` | 68, 69, 72, 73, 84, 85, 87 | stowaway | stone, ceramics, metals, machinery, vehicles: shipped with wood packaging in containers (hitchhikers) |
| `used_tyres` | 401220 | stowaway | breeding sites for container mosquitoes (*Aedes*) |
| `growing_media` | 2703, 3101 | contaminant | peat, manure: soil organisms, seeds |

The **legacy** 5 channels of the current model can be reproduced with
`CHANNEL_SET = "legacy"` (note: HS 3821 only exists from HS 2007, so the
legacy `microbial_risk` channel contains 3002 only in HS 1992).

## Pipeline at a glance

| Step | What happens | Main output |
|---|---|---|
| 0 | Configuration, sources, channel definitions, helpers | — |
| 1 | BACI files and country codes | `baci_codes` |
| 2 | Locations and allocation weights (BACI country → SInAS locations) | `W` |
| 3 | Price deflator (constant US dollars) | `deflator` |
| 4 | Read every year, assign products to channels, aggregate (cached) | `yearly` |
| 5 | Mean annual flows per window (full period and recent) | `flows` |
| 6 | Re-map to SInAS locations | `tensors` |
| 7 | QA | `qa` |
| 8 | Write outputs | files in `data/` |

## Outputs (written to `data/`)

| File | Content |
|---|---|
| `trade_channels.pt` | `trade_tensor` (channels × exporter × importer, log1p of mean annual value), `channels_order`, `locations_order` (same keys as before) + value and quantity tensors for both windows, channel definitions, years and allocation table |
| `trade_flows_long.csv` | All non-zero location → location flows per channel (mean annual value and tonnes, both windows) |
| `trade_location_profiles.csv` | Per location and channel: total imports and exports |
| `trade_location_mapping.csv` | Each SInAS location: ISO3, BACI code(s) used and allocation share |
| `trade_qa.json` | QA report |

The previous `trade_channels.pt` is kept as `trade_channels_previous.pt`.

## Design rules (read before editing)

1. **One nomenclature for all years** (HS 1992), so product codes mean
   the same thing in 1995 and 2024.
2. **Mean annual flows**: the sum over the years of a window divided by the
   number of years (years without trade count as zero). Two windows: the
   full period (`YEARS`) and the recent decade (`RECENT_YEARS`).
3. **Constant prices**: values are deflated to `BASE_YEAR` US dollars with
   the US GDP deflator (World Bank), so 1995 and 2024 dollars are
   comparable. Quantities (tonnes) need no deflation.
4. **BACI countries → SInAS locations**: a location with its own BACI code
   (Greenland, Hong Kong, Bermuda, …) gets exactly that trade. Locations
   without one (Hawaii, Canary Islands, Puerto Rico, …) share their
   country's trade **in proportion to population** (`ALLOCATION`), instead
   of each receiving a full copy as before. Trade *within* a country (e.g.
   mainland USA → Hawaii) is domestic and not in BACI, so it is zero.
5. The model applies its own scaling; `trade_tensor` is `log1p(value)`
   exactly as before so existing code keeps working.

## How to run

1. Install (once): `pip install pandas numpy torch tqdm country_converter`
2. Download BACI HS92 (Step 0) into `../data/` — the zip can stay zipped.
3. Run notebook 01 first (master table with ISO3 codes and population).
4. *Run All*. The first run reads ~30 yearly files (≈ 20–40 min); the
   per-year aggregates are cached, so changing windows, allocation or
   output options afterwards takes seconds. Changing channel definitions
   recomputes the cache automatically.

## Step 0 · Data sources and configuration

| Input | Source | How to obtain |
|---|---|---|
| `BACI_HS92_V202601.zip` | CEPII BACI, version 202601 (January 2026), HS 1992, 1995–2024 — Gaulier & Zignago (2010), *BACI: International Trade Database at the Product-Level. The 1994–2007 Version*, CEPII Working Paper 2010-23 | **Manual download** (free, CC licence; registration may be asked): BACI page <https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=37> → *HS92* zip. Documentation: <https://www.cepii.fr/DATA_DOWNLOAD/baci/doc/baci_webpage.html>. Put the zip (or the extracted folder) in `data/`. |
| `complete_country_features_master.csv` | Notebook 01 (ISO3 codes, population density and area for the allocation) | run notebook 01 |
| US GDP deflator | World Bank, World Development Indicators, `NY.GDP.DEFL.ZS` | **automatic** (World Bank API); without internet the values stay in current dollars (a warning is shown) |

BACI columns: `t` year, `i` exporter, `j` importer, `k` HS 6-digit product,
`v` value in **thousands of current US dollars**, `q` quantity in **metric
tonnes**. BACI reconciles exporter and importer declarations, so each flow
appears once.

In [1]:
import hashlib
import io
import json
import urllib.request
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

c:\Users\simon\Documents\GitHub\horizon-scanner\.venv\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


### Configuration

In [2]:
# ---------------------------------------------------------------- paths
DATA = Path.cwd().parent / "data"
BACI_VERSION = "202601"
BACI_HS = "HS92"
BACI_SOURCE = DATA / f"BACI_{BACI_HS}_V{BACI_VERSION}.zip"  # zip or folder
MASTER_CSV = DATA / "complete_country_features_master.csv"
CACHE_DIR = DATA / "cache" / "baci"

# ---------------------------------------------------------------- outputs
TRADE_OUT = DATA / "trade_channels.pt"
LONG_OUT = DATA / "trade_flows_long.csv"
PROFILE_OUT = DATA / "trade_location_profiles.csv"
MAPPING_OUT = DATA / "trade_location_mapping.csv"
QA_OUT = DATA / "trade_qa.json"

# ---------------------------------------------------------------- choices
YEARS = list(range(1995, 2025))  # full window (HS92 starts in 1995)
RECENT_YEARS = list(range(2015, 2025))  # recent window
CHANNEL_SET = "pathway"  # "pathway" (11 channels) or "legacy" (old 5)
ALLOCATION = "population"  # "population", "area" or "full_copy" (old)
DEFLATE = True  # constant BASE_YEAR US dollars
BASE_YEAR = 2020
TENSOR_SOURCE = ("value", "full")  # which flows become `trade_tensor`

if not BACI_SOURCE.exists():
    folder = BACI_SOURCE.with_suffix("")
    BACI_SOURCE = folder if folder.exists() else BACI_SOURCE
print(f"[{'OK ' if BACI_SOURCE.exists() else 'MISSING'}] {BACI_SOURCE}")
print(f"[{'OK ' if MASTER_CSV.exists() else 'MISSING'}] {MASTER_CSV}")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\BACI_HS92_V202601.zip
[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\complete_country_features_master.csv


### Channel definitions

Each channel lists HS code **prefixes** to include and, optionally, to
exclude. A product is assigned to the **first** channel (in this order)
whose include list matches and whose exclude list does not.

In [3]:
PATHWAY_CHANNELS = {
    "plants_for_planting": dict(
        include=["0601", "0602", "1209"], pathway="escape/contaminant"),
    "cut_flowers_foliage": dict(
        include=["0603", "0604"], pathway="contaminant"),
    "live_terrestrial_animals": dict(
        include=["0101", "0102", "0103", "0104", "0105", "0106"],
        pathway="escape/release"),
    "live_aquatic_animals": dict(
        include=["0301", "03062", "0307"], pathway="escape/contaminant"),
    "fresh_fruit_vegetables": dict(
        include=[f"07{i:02d}" for i in range(1, 10)] + ["0714"]
        + [f"08{i:02d}" for i in range(1, 11)], pathway="contaminant"),
    "grain_seeds_legumes": dict(
        include=["10", "12", "0713"], exclude=["1209"],
        pathway="contaminant"),
    "roundwood_sawnwood": dict(
        include=["4401", "4403", "4404", "4406", "4407"],
        pathway="contaminant"),
    "wood_packaging": dict(include=["4415", "4416"], pathway="stowaway"),
    "containerised_goods": dict(
        include=["68", "69", "72", "73", "84", "85", "87"],
        pathway="stowaway"),
    "used_tyres": dict(include=["401220"], pathway="stowaway"),
    "growing_media": dict(include=["2703", "3101"], pathway="contaminant"),
}
LEGACY_CHANNELS = {
    "flora_and_timber": dict(include=["06", "44"], pathway="legacy"),
    "aquatic_fauna": dict(include=["03"], pathway="legacy"),
    "terrestrial_pets": dict(include=["01"], pathway="legacy"),
    "agri_hitchhikers": dict(include=["07", "08", "10", "12"],
                             pathway="legacy"),
    "microbial_risk": dict(include=["3002", "3821"], pathway="legacy"),
}
CHANNELS = PATHWAY_CHANNELS if CHANNEL_SET == "pathway" else LEGACY_CHANNELS
CHANNEL_NAMES = list(CHANNELS)
CH = len(CHANNEL_NAMES)
CHANNEL_HASH = hashlib.md5(json.dumps(CHANNELS, sort_keys=True)
                           .encode()).hexdigest()[:8]


def channel_of(code6):
    """Channel index of a 6-digit HS code (-1 if in no channel)."""
    for idx, spec in enumerate(CHANNELS.values()):
        if code6.startswith(tuple(spec["include"])) and not \
                code6.startswith(tuple(spec.get("exclude", ["#"]))):
            return idx
    return -1


print(f"{CH} channels ({CHANNEL_SET}), definition hash {CHANNEL_HASH}")

11 channels (pathway), definition hash 8d7d33f4


### Territory rules

Two small tables handle the cases where a SInAS location and BACI do not
line up one-to-one. **Check them if your location list changes.**

* `SOVEREIGN_FALLBACK` — used only when a location's ISO3 code has **no**
  BACI code of its own: the location then shares the trade of this
  country (by population). E.g. BACI reports Puerto Rico inside the USA.
* `BACI_CODE_SPLIT` — historical BACI codes of countries that later split
  (Belgium-Luxembourg until 1998, Serbia and Montenegro, Netherlands
  Antilles, former Sudan). Their trade is divided over the successors by
  population.

In [4]:
SOVEREIGN_FALLBACK = {
    # United States
    "PRI": "USA", "VIR": "USA", "GUM": "USA", "ASM": "USA", "MNP": "USA",
    "UMI": "USA",
    # France (overseas departments, collectivities, TAAF)
    "GLP": "FRA", "MTQ": "FRA", "GUF": "FRA", "REU": "FRA", "MYT": "FRA",
    "MAF": "FRA", "BLM": "FRA", "SPM": "FRA", "ATF": "FRA",
    # Netherlands
    "BES": "NLD", "SXM": "NLD", "CUW": "NLD",
    # United Kingdom
    "GGY": "GBR", "JEY": "GBR", "IMN": "GBR", "SGS": "GBR", "IOT": "GBR",
    "PCN": "GBR", "SHN": "GBR",
    # Nordic countries
    "SJM": "NOR", "BVT": "NOR", "ALA": "FIN",
    # Australia / New Zealand
    "CXR": "AUS", "CCK": "AUS", "NFK": "AUS", "HMD": "AUS", "TKL": "NZL",
    # other
    "ESH": "MAR", "XKX": "SRB", "TWN": "S19",  # BACI: "Other Asia, nes"
}
BACI_CODE_SPLIT = {
    58: ["BEL", "LUX"],  # Belgium-Luxembourg (until 1998)
    891: ["SRB", "MNE", "XKX"],  # Serbia and Montenegro
    530: ["CUW", "SXM", "BES", "ABW"],  # Netherlands Antilles
    736: ["SDN", "SSD"],  # Sudan before 2012
}

### Helper functions

In [6]:
def baci_members():
    """Names of the BACI yearly files and the country-code file.

    Works with the zip as downloaded or with an extracted folder.
    """
    if BACI_SOURCE.is_dir():
        names = [p.name for p in BACI_SOURCE.rglob("*.csv")]
    else:
        with zipfile.ZipFile(BACI_SOURCE) as zf:
            names = zf.namelist()
    years = {}
    for n in names:
        base = Path(n).name
        if base.startswith(f"BACI_{BACI_HS}_Y") and base.endswith(".csv"):
            years[int(base.split("_Y")[1][:4])] = n
    codes = next((n for n in names
                  if Path(n).name.startswith("country_codes")), None)
    return years, codes


def open_member(name):
    """Open a file from the BACI zip / folder as a text stream."""
    if BACI_SOURCE.is_dir():
        return open(next(BACI_SOURCE.rglob(Path(name).name)),
                    encoding="utf-8")
    zf = zipfile.ZipFile(BACI_SOURCE)
    return io.TextIOWrapper(zf.open(name), encoding="utf-8")


PRODUCT_CHANNEL = {}  # HS code -> channel index (-1 = no channel)


def read_year(path_in_source):
    """Aggregate one BACI year to (channel, exporter, importer) sums.

    Products are assigned to channels with :func:`channel_of`; the result
    per code is remembered in PRODUCT_CHANNEL so each code is checked once.

    Parameters
    ----------
    path_in_source : str
        Name of the yearly CSV inside the zip / folder.

    Returns
    -------
    pandas.DataFrame
        Columns ch, i, j, v (thousand current USD), q (tonnes).
    """
    parts = []
    with open_member(path_in_source) as fh:
        reader = pd.read_csv(
            fh, dtype={"t": "int16", "i": "int32", "j": "int32",
                       "k": str, "v": "float64", "q": str},
            chunksize=2_000_000)
        for chunk in reader:
            k = chunk["k"].str.strip().str.zfill(6)
            for code in set(k.unique()) - PRODUCT_CHANNEL.keys():
                PRODUCT_CHANNEL[code] = channel_of(code)
            ch = k.map(PRODUCT_CHANNEL)
            keep = ch >= 0
            if not keep.any():
                continue
            sub = pd.DataFrame({
                "ch": ch[keep].astype("int8"),
                "i": chunk.loc[keep, "i"], "j": chunk.loc[keep, "j"],
                "v": chunk.loc[keep, "v"],
                "q": pd.to_numeric(chunk.loc[keep, "q"].str.strip(),
                                   errors="coerce"),
            })
            parts.append(sub.groupby(["ch", "i", "j"], as_index=False)
                         [["v", "q"]].sum(min_count=1))
    if not parts:
        return pd.DataFrame(columns=["ch", "i", "j", "v", "q"])
    return (pd.concat(parts).groupby(["ch", "i", "j"], as_index=False)
            [["v", "q"]].sum(min_count=1))


def us_gdp_deflator():
    """{year: US GDP deflator} from the World Bank API (None if offline)."""
    url = ("https://api.worldbank.org/v2/country/USA/indicator/"
           "NY.GDP.DEFL.ZS?format=json&per_page=200")
    try:
        with urllib.request.urlopen(url, timeout=30) as resp:
            data = json.load(resp)[1]
    except Exception as err:  # offline, API change, ...
        warnings.warn(f"World Bank deflator not available ({err})")
        return None
    return {int(d["date"]): float(d["value"]) for d in data
            if d.get("value") is not None}

## Step 1 · BACI files and country codes

Lists the yearly files present and reads BACI's country table (numeric
code, name, ISO3). Column names differ slightly between BACI versions,
so they are detected.

In [7]:
print("Step 1: BACI files and country codes")
year_files, codes_file = baci_members()
if not year_files or codes_file is None:
    raise FileNotFoundError(f"No BACI_{BACI_HS}_Y*.csv / country_codes "
                            f"file found in {BACI_SOURCE}")
missing_years = sorted(set(YEARS) - set(year_files))
if missing_years:
    warnings.warn(f"BACI years not found (skipped): {missing_years}")
YEARS = [y for y in YEARS if y in year_files]
RECENT_YEARS = [y for y in RECENT_YEARS if y in year_files]
print(f"   years: {YEARS[0]}-{YEARS[-1]} ({len(YEARS)}) | recent: "
      f"{RECENT_YEARS[0]}-{RECENT_YEARS[-1]}")

with open_member(codes_file) as fh:
    baci_codes = pd.read_csv(fh, dtype=str)
cols = {c.lower(): c for c in baci_codes.columns}
code_col = cols.get("country_code")
iso3_col = cols.get("country_iso3") or cols.get("iso_3digit_alpha")
name_col = (cols.get("country_name") or cols.get("country_name_full")
            or cols.get("country_name_abbreviation"))
baci_codes = pd.DataFrame({
    "code": baci_codes[code_col].astype(int),
    "iso3": baci_codes[iso3_col].str.strip().str.upper(),
    "name": baci_codes[name_col]})
print(f"   {len(baci_codes)} BACI country codes")

Step 1: BACI files and country codes
   years: 1995-2024 (30) | recent: 2015-2024
   238 BACI country codes


## Step 2 · Locations and allocation weights

For each SInAS location we need its ISO3 code (master table; name lookup
as fallback) and a size for sharing trade within a country:
**population = population density × polygon area** (both from notebook
01), or area if population is missing.

The allocation matrix **W** (BACI codes × locations) says which share of a
BACI country's trade each location receives:

1. locations whose ISO3 has its own BACI code share that code;
2. locations without one use `SOVEREIGN_FALLBACK`;
3. historical codes (`BACI_CODE_SPLIT`) are shared over their successors.

Within a group the shares are proportional to `ALLOCATION` (population,
area, or 1 each for `"full_copy"`).

In [8]:
print("Step 2: locations and allocation weights")
import country_converter as coco  # noqa: E402

master = pd.read_csv(MASTER_CSV, low_memory=False)
loc = (master.drop_duplicates("location").set_index("location")
       .sort_index())
locations = list(loc.index)
N = len(locations)
iso3 = {name: (str(v).strip().upper() if isinstance(v, str)
               and len(str(v).strip()) == 3 else None)
        for name, v in loc["ISO3"].items()}
need = [name for name, v in iso3.items() if v is None]
if need:
    found = coco.convert(names=need, to="ISO3", not_found=None)
    found = found if isinstance(found, list) else [found]
    iso3.update({n: f for n, f in zip(need, found)
                 if isinstance(f, str)})
population = loc["pop_dens_mean"] * loc["polygon_area_sqkm"]
size = {"population": population.fillna(loc["polygon_area_sqkm"]),
        "area": loc["polygon_area_sqkm"],
        "full_copy": pd.Series(1.0, index=loc.index)}[ALLOCATION]
size = size.fillna(size.median()).clip(lower=1e-9)

baci_iso3 = set(baci_codes["iso3"])
code_of_iso3 = baci_codes.groupby("iso3")["code"].apply(list).to_dict()

# BACI code -> list of location names that share it
groups = {}
mapping_rows = []
for name in locations:
    own = iso3.get(name)
    target, rule = own, "own_code"
    if own not in baci_iso3:
        target, rule = SOVEREIGN_FALLBACK.get(own), "sovereign_fallback"
    if target not in baci_iso3:
        mapping_rows.append((name, own, None, "no_baci_code"))
        continue
    for code in code_of_iso3[target]:
        groups.setdefault(code, []).append(name)
    mapping_rows.append((name, own, target, rule))
for code, successors in BACI_CODE_SPLIT.items():
    members_ = [n for n in locations if iso3.get(n) in successors]
    if members_:
        groups[code] = members_

col_of = {name: i for i, name in enumerate(locations)}
baci_index = {code: r for r, code in enumerate(sorted(groups))}
W = np.zeros((len(baci_index), N))
for code, members_ in groups.items():
    w = size[members_].to_numpy(float)
    W[baci_index[code], [col_of[m] for m in members_]] = (
        w / w.sum() if ALLOCATION != "full_copy" else 1.0)

mapping = pd.DataFrame(mapping_rows, columns=["location", "iso3",
                                              "baci_iso3", "rule"])


def allocation_share(name, target):
    """Share of its BACI country's trade a location receives."""
    for code in code_of_iso3.get(target, []):
        if code in baci_index:
            return W[baci_index[code], col_of[name]]
    return np.nan


mapping["allocation_share"] = [
    allocation_share(n, t)
    for n, t in zip(mapping["location"], mapping["baci_iso3"])]
print(mapping["rule"].value_counts().to_string())
shared = mapping[mapping["allocation_share"] < 0.999]
print(f"   {len(shared)} locations share a BACI country "
      f"(allocation = {ALLOCATION}); e.g.:")
shared.sort_values("allocation_share").head(10)

Step 2: locations and allocation weights
rule
own_code              260
sovereign_fallback     18
no_baci_code           11
   70 locations share a BACI country (allocation = population); e.g.:


,location,iso3,baci_iso3,rule,allocation_share
271,United States Minor Outlying Islands,UMI,USA,sovereign_fallback,1.525496e-07
216,Saint-Martin,MAF,FRA,sovereign_fallback,8.719634e-07
104,Guernsey,GGY,GBR,sovereign_fallback,1.437946e-06
126,Jersey,JEY,GBR,sovereign_fallback,2.034658e-06
144,Lord Howe Islands,AUS,AUS,own_code,2.117358e-06
85,Fernando de Noronha,BRA,BRA,own_code,2.187006e-06
155,Martinique,MTQ,FRA,sovereign_fallback,1.735589e-05
110,Heard Island and McDonald Island,HMD,AUS,sovereign_fallback,1.782342e-05
37,Bouvet Island,BVT,NOR,sovereign_fallback,1.846116e-05
101,Guadeloupe,GLP,FRA,sovereign_fallback,2.561877e-05


## Step 3 · Price deflator

Values are converted to constant `BASE_YEAR` dollars:
`value_real = value_current × deflator[BASE_YEAR] / deflator[year]`, with
the US GDP deflator from the World Bank. If the API is unreachable,
values stay in current dollars and the QA report says so.

In [9]:
print("Step 3: price deflator")
deflator = us_gdp_deflator() if DEFLATE else None
if deflator and BASE_YEAR in deflator:
    price_factor = {y: deflator[BASE_YEAR] / deflator[y] for y in YEARS
                    if y in deflator}
    lacking = [y for y in YEARS if y not in price_factor]
    for y in lacking:  # latest years may not be published yet
        price_factor[y] = price_factor[max(price_factor)]
    print(f"   constant {BASE_YEAR} USD; factor {YEARS[0]}: "
          f"{price_factor[YEARS[0]]:.2f}, {YEARS[-1]}: "
          f"{price_factor[YEARS[-1]]:.2f}; extrapolated: {lacking}")
else:
    price_factor = {y: 1.0 for y in YEARS}
    print("   no deflation (current USD)")

Step 3: price deflator
   constant 2020 USD; factor 1995: 1.57, 2024: 0.84; extrapolated: []


## Step 4 · Read every year (cached)

Each yearly file (~9 million rows) is read in chunks; rows whose product
is in a channel are summed per (channel, exporter, importer). The
aggregate of each year is cached in `data/cache/baci/` under the channel
definition hash, so re-runs skip the slow part. Changing the channel
definitions gives a new hash and recomputes.

In [10]:
print("Step 4: reading BACI years")
yearly = []
for year in tqdm(YEARS, desc="BACI years", unit="year"):
    cache = CACHE_DIR / f"{BACI_HS}_V{BACI_VERSION}_{CHANNEL_HASH}_" \
                        f"{year}.csv.gz"
    if cache.exists():
        agg = pd.read_csv(cache)
    else:
        agg = read_year(year_files[year])
        agg.to_csv(cache, index=False)
    agg["t"] = year
    yearly.append(agg)
yearly = pd.concat(yearly, ignore_index=True)
yearly["v_real"] = yearly["v"] * yearly["t"].map(price_factor)
world = yearly.groupby(["t", "ch"])["v_real"].sum().unstack("ch")
world.columns = [CHANNEL_NAMES[c] for c in world.columns]
print(f"   {len(yearly):,} (year, channel, exporter, importer) rows")
(world / 1e6).round(1).tail(6)  # billion constant USD per year

Step 4: reading BACI years


BACI years:   0%|          | 0/30 [00:00<?, ?year/s]

   2,295,125 (year, channel, exporter, importer) rows


,plants_for_planting,cut_flowers_foliage,live_terrestrial_animals,live_aquatic_animals,fresh_fruit_vegetables,grain_seeds_legumes,roundwood_sawnwood,wood_packaging,containerised_goods,used_tyres,growing_media
t,,,,,,,,,,,
2019,18.9,10.1,24.0,15.9,181.0,230.8,66.9,4.7,7329.7,0.6,3.2
2020,19.5,9.5,23.9,13.3,183.5,251.1,62.9,4.3,6828.2,0.5,2.7
2021,21.4,11.2,24.4,15.1,190.3,297.5,82.3,5.8,7833.5,0.5,3.1
2022,18.8,10.1,22.5,16.2,175.1,323.4,76.5,6.5,7820.5,0.5,3.0
2023,18.6,9.8,25.2,15.2,177.4,291.0,57.4,5.5,7620.5,0.5,2.7
2024,18.3,9.7,24.1,14.6,184.0,250.8,54.6,5.0,7557.4,0.4,2.7


## Step 5 · Mean annual flows per window

For each window: sum over its years ÷ number of years, giving a mean
annual flow in thousand constant USD (`value`) and tonnes (`quantity`)
for each (channel, BACI exporter, BACI importer).

In [11]:
print("Step 5: mean annual flows")
WINDOWS = {"full": YEARS, "recent": RECENT_YEARS}
flows = {}
for wname, years in WINDOWS.items():
    sub = yearly[yearly["t"].isin(years)]
    agg = (sub.groupby(["ch", "i", "j"])[["v_real", "q"]].sum()
           / len(years))
    flows[wname] = agg.reset_index()
    print(f"   {wname}: {len(years)} years, {len(agg):,} flows")

Step 5: mean annual flows
   full: 30 years, 197,050 flows
   recent: 10 years, 152,055 flows


## Step 6 · Re-map to SInAS locations

For every channel: `F_locations = Wᵀ · F_BACI · W`, i.e. each BACI flow is
shared over the exporting and importing locations by their allocation
shares. BACI codes that map to no location are reported in the QA.

In [12]:
print("Step 6: re-map to SInAS locations")
tensors = {}
lost = {}
for wname, df in flows.items():
    for var, col in (("value", "v_real"), ("quantity", "q")):
        T = np.zeros((CH, N, N), dtype=np.float64)
        ok = df["i"].isin(baci_index) & df["j"].isin(baci_index)
        lost[f"{wname}_{var}"] = float(df.loc[~ok, col].sum()
                                       / max(df[col].sum(), 1e-9))
        sub = df[ok]
        for c in range(CH):
            s = sub[sub["ch"] == c]
            F = np.zeros((len(baci_index), len(baci_index)))
            np.add.at(F, (s["i"].map(baci_index).to_numpy(),
                          s["j"].map(baci_index).to_numpy()),
                      s[col].fillna(0).to_numpy())
            T[c] = W.T @ F @ W
        for c in range(CH):  # no trade of a location with itself
            np.fill_diagonal(T[c], 0.0)
        tensors[(var, wname)] = T.astype(np.float32)
print("   share of BACI flows (value) not mapped to any location:",
      {k: round(v, 4) for k, v in lost.items() if "value" in k})
unmapped_codes = sorted(set(yearly["i"]) - set(baci_index))
print("   unmapped BACI codes:",
      baci_codes[baci_codes["code"].isin(unmapped_codes)]["name"]
      .tolist()[:20])

Step 6: re-map to SInAS locations
   share of BACI flows (value) not mapped to any location: {'full_value': 0.0008, 'recent_value': 0.0}
   unmapped BACI codes: ['Southern African Customs Union (...1999)', 'US Misc. Pacific Isds']


## Step 7 · Quality assurance

* world totals per channel per year (should be smooth, rising over time;
  jumps point to coding problems);
* share of trade lost to unmapped BACI codes;
* allocation table (who shares a country's trade, and how much);
* a sanity look at the main partners of one location (`CHECK_LOCATION`).

In [13]:
print("Step 7: QA")
CHECK_LOCATION = "Belgium"
value_full = tensors[("value", "full")]
qa = {
    "baci_version": BACI_VERSION, "nomenclature": BACI_HS,
    "years_full": [YEARS[0], YEARS[-1]],
    "years_recent": [RECENT_YEARS[0], RECENT_YEARS[-1]],
    "channel_set": CHANNEL_SET, "channels": CHANNEL_NAMES,
    "allocation": ALLOCATION,
    "deflated_to": BASE_YEAR if deflator else None,
    "share_lost_unmapped": lost,
    "unmapped_baci_codes": baci_codes[baci_codes["code"].isin(
        unmapped_codes)][["code", "name"]].to_dict("records"),
    "locations_without_trade": mapping.loc[
        mapping["rule"] == "no_baci_code", "location"].tolist(),
    "world_value_bn_usd_by_year": (world / 1e6).round(3).to_dict(),
}
if CHECK_LOCATION in col_of:
    c = col_of[CHECK_LOCATION]
    imports = value_full[:, :, c].sum(axis=0)
    top = np.argsort(imports)[::-1][:8]
    print(f"   top exporters to {CHECK_LOCATION} (all channels, "
          "mean annual, M USD):")
    print(pd.Series(imports[top] / 1e3,
                    index=[locations[i] for i in top]).round(1)
          .to_string())
print(f"   locations without any trade: {qa['locations_without_trade']}")

Step 7: QA
   top exporters to Belgium (all channels, mean annual, M USD):
Germany                     24253.699219
Netherlands                 16237.000000
France                      16045.299805
Japan                        6697.799805
United Kingdom               6657.799805
Sweden                       4836.399902
China                        4396.200195
United States of America     4164.600098
   locations without any trade: ['Akrotiri and Dhekelia', 'Antarctica', 'Caspian Sea', 'Faroe Islands', 'Kosovo', 'Liechtenstein', 'Monaco', 'Northern Cyprus', 'Paracel Islands', 'Spratly Islands', 'Vatican City']


## Step 8 · Write outputs

`trade_channels.pt` keeps the keys the model reads (`trade_tensor`,
`channels_order`, `locations_order`); `trade_tensor` is
`log1p(mean annual value)` for `TENSOR_SOURCE`. **The model's trade cell
must use `channels_order` from this file** instead of its hard-coded
5-channel mapping when `CHANNEL_SET = "pathway"`.

In [14]:
print("Step 8: writing outputs")
if TRADE_OUT.exists():
    TRADE_OUT.replace(TRADE_OUT.with_name("trade_channels_previous.pt"))
main = tensors[TENSOR_SOURCE]
torch.save({
    "trade_tensor": torch.from_numpy(np.log1p(main)),
    "channels_order": CHANNEL_NAMES,
    "locations_order": locations,
    "tensor_source": list(TENSOR_SOURCE),
    "units": {"value": f"thousand constant {BASE_YEAR} USD per year"
              if deflator else "thousand current USD per year",
              "quantity": "tonnes per year"},
    **{f"{var}_{w}": torch.from_numpy(t)
       for (var, w), t in tensors.items()},
    "channel_definitions": CHANNELS,
    "years": {k: v for k, v in WINDOWS.items()},
    "allocation": ALLOCATION,
    "allocation_matrix": torch.from_numpy(W.astype(np.float32)),
    "baci_codes_order": sorted(baci_index, key=baci_index.get),
}, TRADE_OUT)

rows = []
for c, ch_name in enumerate(CHANNEL_NAMES):
    e, i = np.nonzero(value_full[c] + tensors[("quantity", "full")][c])
    rows.append(pd.DataFrame({
        "exporter": np.array(locations)[e],
        "importer": np.array(locations)[i], "channel": ch_name,
        **{f"{var}_{w}": tensors[(var, w)][c][e, i]
           for (var, w) in tensors}}))
long = pd.concat(rows, ignore_index=True)
long.to_csv(LONG_OUT, index=False)

profiles = []
for c, ch_name in enumerate(CHANNEL_NAMES):
    for (var, w), t in tensors.items():
        profiles.append(pd.DataFrame({
            "location": locations, "channel": ch_name,
            "variable": var, "window": w,
            "imports": t[c].sum(axis=0), "exports": t[c].sum(axis=1)}))
pd.concat(profiles).to_csv(PROFILE_OUT, index=False)
mapping.to_csv(MAPPING_OUT, index=False)
with open(QA_OUT, "w") as fh:
    json.dump(qa, fh, indent=2, default=str)
print(f"   {TRADE_OUT.name} | {LONG_OUT.name} ({len(long):,} rows) | "
      f"{PROFILE_OUT.name} | {MAPPING_OUT.name} | {QA_OUT.name}\nDONE")

Step 8: writing outputs
   trade_channels.pt | trade_flows_long.csv (362,516 rows) | trade_location_profiles.csv | trade_location_mapping.csv | trade_qa.json
DONE
